# STA 9890 Prediction Competition Report

## 1. Background
This project aims to predict student performance on standardized assessments across New York State public schools. These outcomes are widely used to evaluate school effectiveness and identify disparities among student populations. Understanding the factors associated with student performance is essential for improving educational outcomes and informing policy decisions.

---

## 2. Literature Review
Prior research shows that student academic performance is influenced by a combination of socioeconomic, demographic, and institutional factors. Socioeconomic disadvantage is strongly associated with lower academic outcomes (Sirin, 2005). In addition, school- and district-level characteristics, such as funding, teacher quality, and resource availability, play a significant role in shaping student achievement (Hanushek & Rivkin, 2010). Furthermore, systematic differences in performance across demographic subgroups, including gender and socioeconomic classification, have been widely documented (Reardon, 2011).  

Motivated by these findings, this project incorporates school-level, district-level, and subgroup-level variables to improve prediction accuracy and capture the multi-level structure of educational outcomes.

---

## 3. Goal
The objective of this project is to predict:

**PERCENT_PROFICIENT**

for each (school, subgroup, assessment) combination in the test dataset.

---

## 4. Data
The dataset includes multiple sources of information:
- School-level data (demographics, funding, and class size)
- District-level data (graduation outcomes and funding measures)
- Training data (~144,000 observations with the target variable)
- Test data (~48,000 observations without the target variable)

Each observation corresponds to a unique combination of school, subgroup, and assessment.

---

## 5. Methodology
The modeling approach follows a progressive strategy:
- Begin with simple baseline models (mean, grouped averages)
- Incorporate structured features (county, subgroup, assessment)
- Extend to more advanced models (e.g., regression, tree-based models)
- Evaluate performance using Mean Squared Error (MSE)

The modeling approach is guided by prior research and follows a progressive strategy. First, simple baseline models are constructed, such as predicting the overall mean proficiency or group-level averages based on county, subgroup, and assessment. These models provide a benchmark for evaluating performance.

Next, more structured models are developed by incorporating features derived from school- and district-level data, capturing the influence of socioeconomic conditions, institutional resources, and demographic characteristics. More advanced models may be explored to capture nonlinear relationships and interactions among variables.

Model performance is evaluated using Mean Squared Error (MSE), which measures the average squared difference between predicted and observed proficiency rates.

Missing data is addressed through appropriate imputation strategies. Variables with low levels of missingness are imputed using summary statistics, while variables with high levels of missingness are treated cautiously or excluded. For district-level variables with structured missingness, imputation and indicator variables are used to preserve information about missing patterns.

---

## 6. Data Exploration
(Your plots, histograms, and interpretation here)

---

## 7. Model 1: Baseline Model
(Mean or simple grouped model explanation + results)

---

## 8. Model 2: Improved Model
(County/subgroup/assessment model or your own improvement)

---

## 9. Results
(Compare models, discuss performance, and key insights)

---

## 10. Conclusion
Summarize findings, discuss limitations, and suggest possible improvements for future work.

---

## 11. References

Sirin, S. R. (2005).  
Socioeconomic status and academic achievement: A meta-analytic review of research. *Review of Educational Research*, 75(3), 417–453.  

Hanushek, E. A., & Rivkin, S. G. (2010).  
Generalizations about using value-added measures of teacher quality. *American Economic Review*, 100(2), 267–271.  

Reardon, S. F. (2011).  
The widening academic achievement gap between the rich and the poor: New evidence and possible explanations. In *Whither Opportunity? Rising Inequality, Schools, and Children’s Life Chances* (pp. 91–116). Russell Sage Foundation.

merged data: 

- The school-level dataset fully covers all schools in the training data, while the district-level dataset lacks information for a subset of districts, resulting in missing values after merging. Missing diploma data is due to incomplete district-level reporting, not school-level issues.
- missing data is uneven, but most features are usable with proper cleaning.
A few columns have very high missing values (>60–90%) → likely not reliable.
Some columns have moderate missing (~11%), mainly diploma-related → usable but need handling: fill with group mean OR add “missing indicator”
Most columns have low missing (<5%) → safe to keep and easy to fill.



In [1]:
import os
os.getcwd()

'/Users/xjw/STA9890-2026-Prediction Competition/notebooks'

In [2]:
import pandas as pd

train = pd.read_csv("../data/scores_training.csv")
test = pd.read_csv("../data/scores_test.csv")
school = pd.read_csv("../data/school_covariates.csv")
district = pd.read_csv("../data/district_covariates.csv")

print("train:", train.shape)
print("test:", test.shape)
print("school:", school.shape)
print("district:", district.shape)

train: (144921, 6)
test: (48307, 5)
school: (4754, 52)
district: (674, 6)


In [3]:
print("TRAIN:")
print(train.head())

print("\nTEST:")
print(test.head())

TRAIN:
  ASSESSMENT_ID    SCHOOL               SUBGROUP_NAME    ASSESSMENT_NAME  \
0  3b6deef53665  e037d064  Economically Disadvantaged               ELA4   
1  962a3bfbfe84  5f633522                        Male               ELA6   
2  ffe086287b6e  2b76539e                All Students  Regents Algebra I   
3  e6f80847409d  31289ced                All Students              MATH6   
4  676cc6d81961  219db6e5                All Students               ELA8   

   N_STUDENTS  PERCENT_PROFICIENT  
0         130                  38  
1          23                  65  
2          94                  65  
3          56                  52  
4         134                  46  

TEST:
  ASSESSMENT_ID    SCHOOL                   SUBGROUP_NAME  \
0  8af5e0382a81  a49eed66  Not Economically Disadvantaged   
1  e1591bf8db41  022f98d2      Economically Disadvantaged   
2  547ec44dcea6  255d51d5                    All Students   
3  0e200399fc40  9442d8c4  Not Economically Disadvantaged   
4  c2c40

In [4]:
school.head()

,SCHOOL,DISTRICT,COUNTY,DISTRICT_TYPE,REGION,ATTENDANCE_RATE,LANGUAGE_ARTS_AVERAGE_CLASS_SIZE,MATHEMATICS_AVERAGE_CLASS_SIZE,SCIENCE_AVERAGE_CLASS_SIZE,HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE,...,PERCENT_ASIAN,PERCENT_HISPANIC,PERCENT_WHITE,PERCENT_MULTIRACIAL,PERCENT_WITH_DISABILITIES,PERCENT_ECONOMICALLY_DISADVANTAGED,PERCENT_MIGRANT,PERCENT_HOMELESS,PERCENT_IN_FOSTER_CARE,PERCENT_PARENT_ARMED_FORCES
0,200cb9f4,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",95.0,23.000000,23.000000,21.0,NaN,...,13.0,11.0,43.0,11.0,10.0,36.0,0.0,1.0,0.0,0.0
1,fe6c45ae,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",92.0,12.000000,12.333333,12.0,NaN,...,15.0,22.0,19.0,8.0,15.0,76.0,0.0,4.0,2.0,0.0
2,edd71301,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",91.0,11.666667,11.666667,11.0,NaN,...,17.0,35.0,9.0,7.0,5.0,84.0,0.0,3.0,0.0,0.0
3,f3fc11ad,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",94.0,13.000000,13.000000,15.0,NaN,...,12.0,15.0,38.0,11.0,14.0,48.0,0.0,2.0,0.0,0.0
4,6f19e0a6,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",93.0,19.666667,19.666667,23.0,NaN,...,10.0,17.0,18.0,11.0,13.0,60.0,0.0,6.0,0.0,0.0


In [5]:
district.head()

,DISTRICT,PERCENT_DIPLOMA,PERCENT_NON_DIPLOMA,PERCENT_STILL_ENROLLED,PERCENT_GED,PERCENT_DROPOUT
0,f5a6cc88,69,2,18,0,11
1,0a4984df,90,1,4,0,4
2,56f2c546,96,1,1,0,2
3,e93bd0b0,84,2,7,0,7
4,b469f4b9,79,1,7,0,14


In [6]:
# Count unique schools
train_schools = set(train["SCHOOL"])
school_schools = set(school["SCHOOL"])

print("Total unique schools in TRAIN:", len(train_schools))
print("Total unique schools in SCHOOL dataset:", len(school_schools))

# Check if sets are identical
print("\nAre both datasets using the same schools?")
print(train_schools == school_schools)

# Find differences
only_in_train = train_schools - school_schools
only_in_school = school_schools - train_schools

print("\nSchools in TRAIN but NOT in SCHOOL dataset:", len(only_in_train))
print("Schools in SCHOOL dataset but NOT in TRAIN:", len(only_in_school))

# Show a few examples
print("\nExample schools only in TRAIN:", list(only_in_train)[:5])
print("Example schools only in SCHOOL dataset:", list(only_in_school)[:5])

Total unique schools in TRAIN: 4469
Total unique schools in SCHOOL dataset: 4754

Are both datasets using the same schools?
False

Schools in TRAIN but NOT in SCHOOL dataset: 0
Schools in SCHOOL dataset but NOT in TRAIN: 285

Example schools only in TRAIN: []
Example schools only in SCHOOL dataset: ['5dcfab51', '856cdd83', '24681213', '9cdfd14b', '320b9d6c']


In [7]:
# Extract unique districts
school_districts = set(school["DISTRICT"])
district_districts = set(district["DISTRICT"])

# Basic counts
print("Total districts in SCHOOL dataset:", len(school_districts))
print("Total districts in DISTRICT dataset:", len(district_districts))

# Check if they are identical
print("\nAre both datasets using the same districts?")
print(school_districts == district_districts)

# Find differences
only_in_school = school_districts - district_districts
only_in_district = district_districts - school_districts

print("\nDistricts in SCHOOL but NOT in DISTRICT:", len(only_in_school))
print("Districts in DISTRICT but NOT in SCHOOL:", len(only_in_district))

# Show a few examples (if any)
print("\nExample only in SCHOOL:", list(only_in_school)[:5])
print("Example only in DISTRICT:", list(only_in_district)[:5])

Total districts in SCHOOL dataset: 721
Total districts in DISTRICT dataset: 674

Are both datasets using the same districts?
False

Districts in SCHOOL but NOT in DISTRICT: 47
Districts in DISTRICT but NOT in SCHOOL: 0

Example only in SCHOOL: ['b8a401c3', 'c9b19c7e', '18a67837', '2440845b', 'b3d8bde9']
Example only in DISTRICT: []


In [8]:
# 1. MERGE TRAIN DATA

train_merged = train.merge(school, on="SCHOOL", how="left")
train_merged = train_merged.merge(district, on="DISTRICT", how="left")

print("Train merged shape:", train_merged.shape)

# 2. MERGE TEST DATA

test_merged = test.merge(school, on="SCHOOL", how="left")
test_merged = test_merged.merge(district, on="DISTRICT", how="left")

print("Test merged shape:", test_merged.shape)

Train merged shape: (144921, 62)
Test merged shape: (48307, 61)


In [9]:
import pandas as pd

pd.set_option('display.max_rows', None)

# calculate missing
missing = train_merged.isna().sum()

# create dataframe with count + percent
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': (missing / len(train_merged)) * 100
})

# sort by percentage (more informative)
missing_df = missing_df.sort_values(by='missing_percent', ascending=False)

# round percentage
missing_df['missing_percent'] = missing_df['missing_percent'].round(2)

missing_df

,missing_count,missing_percent
TEACHER_TURNOVER_RATE,134136,92.56
KINDERGARTEN_AVERAGE_CLASS_SIZE,103467,71.40
GRADE_1_AVERAGE_CLASS_SIZE,102899,71.00
GRADE_2_AVERAGE_CLASS_SIZE,102779,70.92
HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE,89314,61.63
PERCENT_DROPOUT,16061,11.08
PERCENT_NON_DIPLOMA,16061,11.08
PERCENT_GED,16061,11.08
PERCENT_STILL_ENROLLED,16061,11.08
PERCENT_DIPLOMA,16061,11.08


In [11]:
train_merged.columns

Index(['ASSESSMENT_ID', 'SCHOOL', 'SUBGROUP_NAME', 'ASSESSMENT_NAME',
       'N_STUDENTS', 'PERCENT_PROFICIENT', 'DISTRICT', 'COUNTY',
       'DISTRICT_TYPE', 'REGION', 'ATTENDANCE_RATE',
       'LANGUAGE_ARTS_AVERAGE_CLASS_SIZE', 'MATHEMATICS_AVERAGE_CLASS_SIZE',
       'SCIENCE_AVERAGE_CLASS_SIZE',
       'HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE',
       'GRADE_1_AVERAGE_CLASS_SIZE', 'GRADE_2_AVERAGE_CLASS_SIZE',
       'KINDERGARTEN_AVERAGE_CLASS_SIZE', 'PERCENT_FREE_LUNCH',
       'PERCENT_REDUCED_LUNCH', 'NUMBER_OF_TEACHERS', 'NUMBER_OF_COUNSELORS',
       'NUMBER_OF_SOCIAL_WORKERS', 'TEACHER_TURNOVER_RATE',
       'PERCENT_OF_STUDENTS_SUSPENDED', 'N_PUPILS',
       'FEDERAL_FUNDING_PER_PUPIL', 'LOCAL_FUNDING_PER_PUPIL', 'PRE_K', 'K',
       'GRADE_01', 'GRADE_02', 'GRADE_03', 'GRADE_04', 'GRADE_05', 'GRADE_06',
       'GRADE_07', 'GRADE_08', 'GRADE_09', 'GRADE_10', 'GRADE_11', 'GRADE_12',
       'PERCENT_MALE', 'PERCENT_FEMALE', 'PERCENT_ENGLISH_LANGUAGE_LEANERS',
   

In [12]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# =========================
# 2. DEFINE FEATURES
# =========================

categorical_features = [
    "SUBGROUP_NAME",
    "ASSESSMENT_NAME",
    "COUNTY"
]

ses_features = [
    "PERCENT_FREE_LUNCH",
    "PERCENT_REDUCED_LUNCH",
    "PERCENT_ECONOMICALLY_DISADVANTAGED"
]

demographic_features = [
    "PERCENT_BLACK",
    "PERCENT_WHITE",
    "PERCENT_HISPANIC",
    "PERCENT_ASIAN",
    "PERCENT_FEMALE",
    "PERCENT_ENGLISH_LANGUAGE_LEANERS",
    "PERCENT_WITH_DISABILITIES"
]

school_features = [
    "ATTENDANCE_RATE",
    "NUMBER_OF_TEACHERS",
    "NUMBER_OF_COUNSELORS",
    "NUMBER_OF_SOCIAL_WORKERS"
]

funding_features = [
    "FEDERAL_FUNDING_PER_PUPIL",
    "LOCAL_FUNDING_PER_PUPIL"
]

optional_features = [
    "N_STUDENTS",
    "N_PUPILS"
]

features = (
    categorical_features +
    ses_features +
    demographic_features +
    school_features +
    funding_features +
    optional_features
)

# =========================
# 3. BUILD TRAINING DATA
# =========================

X = train_merged[features].copy()
y = train_merged["PERCENT_PROFICIENT"]

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Fill missing values
X = X.fillna(X.median(numeric_only=True))

# =========================
# 4. TRAIN MODEL
# =========================

model = LinearRegression()
model.fit(X, y)

# =========================
# 5. PREPARE TEST DATA
# =========================

X_test = test_merged[features].copy()

# Encode categorical variables
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns with training data
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# Fill missing values
X_test = X_test.fillna(X.median(numeric_only=True))

# =========================
# 6. MAKE PREDICTIONS
# =========================

preds = model.predict(X_test)

# =========================
# 7. CREATE SUBMISSION FILE
# =========================

submission = test_merged[["ASSESSMENT_ID"]].copy()
submission["PERCENT_PROFICIENT"] = preds

# Save to CSV
submission.to_csv("submission.csv", index=False)

print("Submission file created: submission.csv")

Submission file created: submission.csv
